In [1]:
import pandas as pd

# Load the datasets
movies = pd.read_csv('../data/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/tmdb_5000_credits.csv')

# The credits file uses 'movie_id' while movies uses 'id'. Let's align and merge them.
credits = credits.rename(columns={'movie_id': 'id'})
df = movies.merge(credits, on='id')

# View the columns we now have available
print("Columns:", df.columns.tolist())
df.head(2)

Columns: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average', 'vote_count', 'title_y', 'cast', 'crew']


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title_x,vote_average,vote_count,title_y,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [2]:
import pandas as pd
import ast

# 1. Reload data to reset the notebook state
movies = pd.read_csv('../data/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/tmdb_5000_credits.csv')
credits = credits.rename(columns={'movie_id': 'id'})
df = movies.merge(credits, on='id')

# 2. Extract names safely
def extract_names(x):
    try:
        return [i['name'] for i in ast.literal_eval(str(x))]
    except:
        return []

# 3. Apply to columns
for col in ['genres', 'keywords', 'cast', 'production_companies']:
    df[col] = df[col].apply(extract_names)

df = df.rename(columns={'title_x': 'title'})
print(df[['title', 'genres', 'cast']].head(2))

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   

                                          genres  \
0  [Action, Adventure, Fantasy, Science Fiction]   
1                   [Adventure, Fantasy, Action]   

                                                cast  
0  [Sam Worthington, Zoe Saldana, Sigourney Weave...  
1  [Johnny Depp, Orlando Bloom, Keira Knightley, ...  


In [3]:
# Fetch Director
def fetch_director(x):
    try:
        for i in ast.literal_eval(str(x)):
            if i['job'] == 'Director':
                return i['name']
        return "Unknown"
    except:
        return "Unknown"

df['director'] = df['crew'].apply(fetch_director)

# Extract release year for the Era filter
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year

print(df[['title', 'director', 'release_year']].head(2))

                                      title        director  release_year
0                                    Avatar   James Cameron        2009.0
1  Pirates of the Caribbean: At World's End  Gore Verbinski        2007.0


In [4]:
# Remove spaces so tags are unique entities
def collapse(lst):
    return [i.replace(" ", "") for i in lst]

for col in ['genres', 'keywords', 'cast']:
    df[col] = df[col].apply(collapse)

# Process director and overview
df['director_tag'] = df['director'].apply(lambda x: [x.replace(" ", "")] if x != "Unknown" else [])
df['overview'] = df['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])

# Combine into a single tags column
df['tags'] = df['overview'] + df['genres'] + df['keywords'] + df['cast'] + df['director_tag']
df['tags'] = df['tags'].apply(lambda x: " ".join(x).lower())

print(df[['title', 'tags']].head(2))

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   

                                                tags  
0  in the 22nd century, a paraplegic marine is di...  
1  captain barbossa, long believed to be dead, ha...  


In [5]:
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer

nltk.download('punkt') # run once
ps = PorterStemmer()

def stem(text):
    return " ".join([ps.stem(word) for word in text.split()])

df['tags'] = df['tags'].apply(stem)

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(df['tags']).toarray()

print(f"Vector shape: {vectors.shape}")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\suluc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Vector shape: (4803, 5000)


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

print("Calculating similarity matrix...")
similarity = cosine_similarity(vectors)

def test_recommender(movie_title):
    try:
        # Find the index of the movie
        movie_index = df[df['title'] == movie_title].index[0]
        distances = similarity[movie_index]
        
        # Sort distances in descending order, grab top 5
        movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
        
        print(f"\nTop matches for '{movie_title}':")
        for i in movies_list:
            print("-", df.iloc[i[0]].title)
    except IndexError:
        print("Movie not found.")

# Let's test it!
test_recommender('Avatar')
test_recommender('The Dark Knight')

Calculating similarity matrix...

Top matches for 'Avatar':
- Aliens vs Predator: Requiem
- Predator
- Battle: Los Angeles
- Independence Day
- Titan A.E.

Top matches for 'The Dark Knight':
- Batman Begins
- The Dark Knight Rises
- Batman Returns
- Amidst the Devil's Wings
- Batman Forever


In [9]:
# 1. Define the 8 CineMatch Moods with descriptive seed keywords
mood_profiles = {
    "Want to cry": "drama romance emotional tragic tearjerker heartbreak grief",
    "Need to laugh": "comedy hilarious funny laugh spoof satire stupid goofy",
    "Edge of my seat": "thriller suspense tension twist mystery crime intense",
    "Turn my brain off": "action explosion blockbuster popcorn dumb fun giant",
    "Feel inspired": "biography inspiring triumph success underdog true story hope",
    "Get scared": "horror scary ghost demon slasher blood nightmare terror",
    "Fall in love": "romance romantic comedy love relationship wedding couple",
    "Go on a journey": "adventure fantasy epic quest journey magic world discover"
}

# 2. Function to recommend by Mood
def recommend_by_mood(mood_name):
    if mood_name not in mood_profiles:
        return "Mood not found."
    
    # Stem the seed text just like we did the movies
    seed_text = stem(mood_profiles[mood_name])
    
    # Transform the text into a vector (using the existing cv object)
    mood_vector = cv.transform([seed_text]).toarray()
    
    # Calculate similarity between the mood vector and all movie vectors
    from sklearn.metrics.pairwise import cosine_similarity
    mood_distances = cosine_similarity(mood_vector, vectors)[0]
    
    # Get top 5 matches
    movies_list = sorted(list(enumerate(mood_distances)), reverse=True, key=lambda x: x[1])[0:5]
    
    print(f"\nTop matches for mood '{mood_name}':")
    for i in movies_list:
        print(f"- {df.iloc[i[0]].title} (Match Score: {round(i[1], 2)})")

# Test it out!
recommend_by_mood("Get scared")
recommend_by_mood("Want to cry")


Top matches for mood 'Get scared':
- Tales from the Crypt: Demon Knight (Match Score: 0.29)
- The Horror Network Vol. 1 (Match Score: 0.29)
- Jennifer's Body (Match Score: 0.24)
- Friday the 13th Part III (Match Score: 0.24)
- Teeth and Blood (Match Score: 0.24)

Top matches for mood 'Want to cry':
- Love Me Tender (Match Score: 0.33)
- Two Lovers (Match Score: 0.3)
- Code 46 (Match Score: 0.3)
- Chiamatemi Francesco - Il Papa della gente (Match Score: 0.29)
- The Heart of Me (Match Score: 0.29)


In [11]:
import pickle
import os

# Create a folder to hold our exported model files
os.makedirs('../models', exist_ok=True)

# 1. Export the movie data (converting to dict makes it easier for FastAPI to handle)
pickle.dump(df.to_dict(), open('../models/movie_dict.pkl', 'wb'))

# 2. Export the trained Vectorizer
pickle.dump(cv, open('../models/vectorizer.pkl', 'wb'))

# 3. Export the pre-calculated Vectors
pickle.dump(vectors, open('../models/vectors.pkl', 'wb'))

print("Success! Model files exported.")

Success! Model files exported.
